<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E1_KMeans_K_Optimo_Mall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E1 · Repaso aplicado: K óptimo - Análisis cluster

## Introducción

Repasamos **K-Means** de principio a fin sobre un dataset de clientes real y elegimos el
**número de grupos (K)** con dos herramientas: el **método del codo** y el **Silhouette Score**.

Recuerda el bucle de K-Means: eliges K, colocas K centroides (con **k-means++**), asignas cada
punto a su centroide más cercano y los recalculas hasta que no se mueven. Como mide distancias,
**escalar es obligatorio**. Y K no lo decide el algoritmo: lo eliges tú con criterio.

## Objetivos del ejercicio

- Aplicar K-Means con `k-means++` sobre datos escalados.
- Elegir K con el **método del codo** (inercia) y el **Silhouette Score** (K=2..6).
- Justificar el K elegido y visualizar los grupos.

## Descripción del dataset (Mall Customers)

Usamos **Mall Customers**, un dataset real muy conocido de segmentación: 200 clientes de un
centro comercial con su edad (`Age`), sus ingresos anuales (`Annual Income (k$)`) y una
puntuación de gasto (`Spending Score (1-100)`). Se carga directamente desde una URL pública.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### 2. Cargar el dataset

In [ ]:
import pandas as pd

def cargar_mall():
    # Mall Customers: dataset clasico y REAL de segmentacion (200 clientes).
    # Columnas: CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100).
    urls = [
        "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv",
        "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv",
    ]
    for u in urls:
        try:
            return pd.read_csv(u).rename(columns={"Genre": "Gender"})
        except Exception:
            continue
    raise RuntimeError("No se pudo descargar Mall Customers")

In [ ]:
df = cargar_mall()
print("Forma:", df.shape)
df.head()

### 3. Escalar (paso obligatorio en clustering)

In [ ]:
# Agrupamos por las dos variables de COMPORTAMIENTO (ingreso y gasto). La edad la
# dejamos fuera del clustering y la usaremos para perfilar en E4.
cols = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[cols]
print("Escalas distintas, por eso escalamos:")
print(X.describe().loc[["min", "max"]].round(1))

X_esc = StandardScaler().fit_transform(X)

### 4. Método del codo (inercia)

In [ ]:
Ks = range(2, 7)
inercias = [KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit(X_esc).inertia_
            for k in Ks]

plt.figure(figsize=(8, 4))
plt.plot(list(Ks), inercias, marker="o")
plt.title("Método del codo")
plt.xlabel("K"); plt.ylabel("Inercia")
plt.tight_layout(); plt.show()

### 5. Silhouette Score (K=2..6)

In [ ]:
sils = []
for k in Ks:
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    sils.append(silhouette_score(X_esc, lab))
    print(f"K={k}: silhouette = {sils[-1]:.3f}")

mejor_k = list(Ks)[int(np.argmax(sils))]
print(f"\nMejor K segun silhouette: {mejor_k}")

### 6. K elegido y visualización

In [ ]:
km = KMeans(n_clusters=mejor_k, init="k-means++", n_init=10, random_state=0)
labels = km.fit_predict(X_esc)

plt.figure(figsize=(7, 6))
sc = plt.scatter(df["Annual Income (k$)"], df["Spending Score (1-100)"], c=labels, cmap="tab10", s=30)
plt.xlabel("Ingreso anual (k$)"); plt.ylabel("Spending Score (1-100)")
plt.title(f"Clientes agrupados con K-Means (K={mejor_k})")
plt.tight_layout(); plt.show()

### 7. Output: K elegido + justificación

In [ ]:
print(f"K ELEGIDO: {mejor_k}")
print("Justificación:")
print(f"  - Silhouette máximo en K={mejor_k} ({max(sils):.3f}).")
print(f"  - En el codo, la inercia deja de bajar con fuerza a partir de ese punto.")
print(f"  - Los grupos son visualmente separables en ingreso vs spending.")

### Reflexión

1. ¿Coinciden el codo y el silhouette en el mismo K?
2. ¿Qué pasaría con los grupos si no hubiéramos escalado?
3. ¿La `Age` ayuda a separar grupos o casi todo lo deciden ingreso y gasto?
4. ¿Qué harías si el negocio te pide menos grupos de los que sugiere el silhouette?